In [1]:
from typing import final

import requests
import json
import pandas as pd
from datetime import datetime as dt
import numpy as np
import sns

import timestamps
from dateutil import parser
import time
#import subprocess
import logging
import os
import sys
import pdb
import matplotlib.pyplot as plt
from datetime import datetime, timezone, timedelta
import matplotlib.dates as mdates
import ast

import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error

In [2]:
#Regular functions may need
# Show all columns when printing a pandas DataFrame
pd.options.display.max_columns = None
pd.set_option('display.max_colwidth', None)

# Craetes a list of the fileds with a file as input
def read_fields(filename):
    fields = []
    fd = open(filename)
    lines = fd.readlines()
    for line in lines:
        fields.append("data."+line[:-1])
    fd.close()
    return fields

def remove_dot_data(fields):
    fields_no_data=[]
    for field in fields:
        field_no_data = field.replace("data.","")
        fields_no_data.append(field_no_data)
    return fields_no_data

def read_token(filename):
    fd = open(filename)
    line = fd.readline()
    # Remove the 'newline' character if found
    if line[len(line)-1] == '\n':
        line = line[:-1]
    fd.close()
    return line


def last_n_hrs(n):
    curr = int(round(time.time() * 1000))
    curr_minus_n = curr - n * 3600 *1000
    return curr_minus_n, curr

def break_in_n_mins(start, end, mins):
    intervals = []
    for i in range(start,end,(mins*60*1000)):
        intervals.append([i, i+(mins*60*1000)])
    return intervals

def get_records(data):
    records=[]
    for record in data['hits']['hits']:
        timestamp = record['_source']['data']['timestamp']
        job_state = record['_source']['data']['job_state']
        records.append([timestamp, job_state])
    return records

def get_records_v2(data):
    # Create a pandas DataFrame with the data retreived
    records=[]
    for record in data['hits']['hits']:
        record= record['_source']['data']
        records.append(record)
    return records


In [3]:
def create_query(query, fields, min_date, max_date):

    # Read the list of fields to retrieve from a file

    fields_no_data = remove_dot_data(fields)

    # Number of records to retreive (max is 10,000)
    num_records = 10000

    data = {
            "size":num_records,
            "query":{
                "bool":{
                    "filter":[
                        {"range":{"metadata.event_timestamp":{"gte":min_date,"lte":max_date,"format":"epoch_millis"}}},
                        {"query_string":{
                            "analyze_wildcard":"true",
                            "query":query
                            }
                        }
                    ]
                }
            },
            "_source":fields,
    }

    data_string = json.dumps(data)
    return data_string

In [4]:
from datetime import datetime

#configuration params
query = 'data.dst_experiment_site:T2_US_UCSD'
fields = []
fields += ['data.tr_timestamp_start', 'data.tr_timestamp_complete'] # start/end of the whole operation (includes preparation/checksum)
fields += ['data.timestamp_tr_st','data.timestamp_tr_comp'] # when the COPY started/finished (** Used to calculate throughput **)
fields += ['data.throughput'] # transferred_volume/(timestamp_tr_comp -timestamp_tr_st)
fields += ['data.transfer_time'] # Time for just the COPY part (timestamp_tr_comp -timestamp_tr_st)
fields += ['data.transferred_volume'] # same as file_size
fields += ['data.transfer_size']
#fields += ['data.tr_bt_transfered'] # DO NOT USE: last reported bytes from PerformanceMarker (missleading)

fields = ['data.dst_hostname', 'data.timestamp_tr_comp', 'data.transferred_volume','data.throughput', 'data.timestamp_tr_st', 'data.t_final_transfer_state','data.dst_url','data.src_experiment_site','data.dst_experiment_site', 'data.src_hostname','data.src_url','data.activity']


In [5]:
# Set start_dt and end_dt
start_dt = datetime(2026, 10, 1, 0, 0, tzinfo=timezone.utc)
end_dt = datetime(2026, 10, 7, 23, 59, tzinfo=timezone.utc)

min_time_ms = int(start_dt.timestamp() * 1000)
max_time_ms = int(end_dt.timestamp() * 1000)

days = []
intervals = break_in_n_mins(min_time_ms, max_time_ms, 60)
days = intervals

#don't touch
token = "ADD_TOKEN_HERE"
headers = {'Authorization': 'Bearer '+token, 'Content-Type': 'application/json'}
all_records = []

def fetch_all_fts_data(start_ms, end_ms, query, fields_list):
    url = 'https://monit-grafana.cern.ch/api/datasources/proxy/9233/monit_prod_fts_enr_complete*/_search'
    all_records = []
    search_after = None
    batch_count = 0

    print(f"Starting query from {start_dt} to {end_dt}...")

    while True:
        payload = {
            "size": 10000,
            "query": {
                "bool": {
                    "filter": [
                        {"range": {"data.tr_timestamp_start": {"gte": start_ms, "lte": end_ms}}},
                        {"query_string": {"query": query}},
                        {"term": {"data.t_final_transfer_state": "Ok"}},
                        {"range": {"data.throughput": {"gt": 0}}}
                    ],
                    "must_not": [
                        {"term": {"data.activity": "ASO"}},
                        {"term": {"data.transferred_volume": 0}}
                    ]
                }
            },
            "_source": fields_list,
            "sort": [
                {"data.tr_timestamp_start": "asc"},
                {"_id": "asc"}
            ]
        }

        if search_after:
            payload["search_after"] = search_after

        try:
            response = requests.get(url, headers=headers, data=json.dumps(payload))
            response.raise_for_status()
            res_data = response.json()
        except Exception as e:
            print(f"Request failed at batch {batch_count}: {e}")
            break

        hits = res_data.get('hits', {}).get('hits', [])

        if not hits:
            break

        for hit in hits:
            all_records.append(hit["_source"]["data"])


        batch_count += 1
        print(f"Batch {batch_count}: Fetched {len(hits)} records (Total: {len(all_records)})")
        if len(hits) < 10000:
            break

        search_after = hits[-1]["sort"]

    return all_records

In [6]:
df_cleaned = fetch_all_fts_data(min_time_ms, max_time_ms, query, fields)

print(f"\n{len(df_cleaned)} records in total.")

Starting query from 2026-10-01 00:00:00+00:00 to 2026-10-07 23:59:00+00:00...
Batch 1: Fetched 10000 records (Total: 10000)
Batch 2: Fetched 6010 records (Total: 16010)

16010 records in total.


In [7]:
df_cleaned = pd.DataFrame(df_cleaned)
df_cleaned

,src_url,timestamp_tr_comp,dst_hostname,activity,transferred_volume,dst_url,src_experiment_site,timestamp_tr_st,throughput,src_hostname,dst_experiment_site,t_final_transfer_state
0,davs://sbgdcache.in2p3.fr:443/cms/phedex/store/mc/RunIISummer20UL18NanoAODv9/SMS-TChiWZ_genHT-50_genMET-80_TuneCP5_13TeV-madgraphMLM-pythia8/NANOAODSIM/FSMiniUL18_NANOv9_FSUL18_106X_upgrade2018_realistic_v16_L1v1-v2/2560000/5E3CE41D-8CB3-8442-A6D5-E270FB5ECBF7.root,1790813456467,redir.t2.ucsd.edu,Production Output,630775345,davs://redir.t2.ucsd.edu:1095/store/mc/RunIISummer20UL18NanoAODv9/SMS-TChiWZ_genHT-50_genMET-80_TuneCP5_13TeV-madgraphMLM-pythia8/NANOAODSIM/FSMiniUL18_NANOv9_FSUL18_106X_upgrade2018_realistic_v16_L1v1-v2/2560000/5E3CE41D-8CB3-8442-A6D5-E270FB5ECBF7.root,T2_FR_IPHC,1790813444523,5.281106e+07,sbgdcache.in2p3.fr,T2_US_UCSD,Ok
1,davs://grid143.kfki.hu:443/dpm/kfki.hu/home/cms/phedex/store/test/loadtest/source/T2_HU_Budapest/urandom.270MB.file0001,1790813521148,redir.t2.ucsd.edu,Functional Test,270000000,davs://redir.t2.ucsd.edu:1095/store/test/loadtest/source/T2_HU_Budapest/urandom.270MB.file0001,T2_HU_Budapest,1790813506557,1.850456e+07,grid143.kfki.hu,T2_US_UCSD,Ok
2,davs://cmsdcatape.fnal.gov:2880/WAX/11/store/data/Run2026C/SpecialZeroBias28/ALCARECO/LumiPixelsMinBias-PromptReco-v1/000/403/035/00000/55f95cc5-465a-4f3c-beb8-d2423e22bb53.root,1790813707648,redir.t2.ucsd.edu,Analysis TapeRecall,4188973932,davs://redir.t2.ucsd.edu:1095/store/data/Run2026C/SpecialZeroBias28/ALCARECO/LumiPixelsMinBias-PromptReco-v1/000/403/035/00000/55f95cc5-465a-4f3c-beb8-d2423e22bb53.root,T1_US_FNAL,1790813630482,5.428523e+07,cmsdcatape.fnal.gov,T2_US_UCSD,Ok
3,davs://cmsdcatape.fnal.gov:2880/WAX/11/store/data/Run2026C/SpecialZeroBias28/ALCARECO/LumiPixelsMinBias-PromptReco-v1/000/403/035/00000/661f65b9-d733-4929-b1ba-f41b8eb2745a.root,1790813696723,redir.t2.ucsd.edu,Analysis TapeRecall,2615953139,davs://redir.t2.ucsd.edu:1095/store/data/Run2026C/SpecialZeroBias28/ALCARECO/LumiPixelsMinBias-PromptReco-v1/000/403/035/00000/661f65b9-d733-4929-b1ba-f41b8eb2745a.root,T1_US_FNAL,1790813685643,2.360969e+08,cmsdcatape.fnal.gov,T2_US_UCSD,Ok
4,davs://cmsdcatape.fnal.gov:2880/WAX/11/store/data/Run2026C/SpecialZeroBias28/ALCARECO/LumiPixelsMinBias-PromptReco-v1/000/403/080/00000/fbaedaf4-9bf6-455d-be80-bf6c3e62d9d6.root,1790813870054,redir.t2.ucsd.edu,Analysis TapeRecall,3865208287,davs://redir.t2.ucsd.edu:1095/store/data/Run2026C/SpecialZeroBias28/ALCARECO/LumiPixelsMinBias-PromptReco-v1/000/403/080/00000/fbaedaf4-9bf6-455d-be80-bf6c3e62d9d6.root,T1_US_FNAL,1790813817813,7.398802e+07,cmsdcatape.fnal.gov,T2_US_UCSD,Ok
...,...,...,...,...,...,...,...,...,...,...,...,...
16005,davs://se-wbdv.jinr-t1.ru:2880/pnfs/jinr-t1.ru/data/cms/store/mc/RunIII2024Summer24MiniAODv6/WZTo3LNu-TT_TuneCP5_13p6TeV_madgraph-pythia8/MINIAODSIM/150X_mcRun3_2024_realistic_v2-v2/2810000/5bb5d27c-6a21-4c28-bf23-d2a64e53308f.root,1791416862154,redir.t2.ucsd.edu,Data rebalancing,2278184057,davs://redir.t2.ucsd.edu:1095/store/mc/RunIII2024Summer24MiniAODv6/WZTo3LNu-TT_TuneCP5_13p6TeV_madgraph-pythia8/MINIAODSIM/150X_mcRun3_2024_realistic_v2-v2/2810000/5bb5d27c-6a21-4c28-bf23-d2a64e53308f.root,T1_RU_JINR,1791416739591,1.858786e+07,se-wbdv.jinr-t1.ru,T2_US_UCSD,Ok
16006,davs://xrootd2.hepgrid.uerj.br:1094/store/test/loadtest/source/T2_BR_UERJ/urandom.270MB.file0001,1791417070110,redir.t2.ucsd.edu,Functional Test,270000000,davs://redir.t2.ucsd.edu:1095/store/test/loadtest/source/T2_BR_UERJ/urandom.270MB.file0001,T2_BR_UERJ,1791417051937,1.485721e+07,xrootd2.hepgrid.uerj.br,T2_US_UCSD,Ok
16007,davs://webdav.recas.ba.infn.it:8443/cms/store/mc/Run3Summer22EENanoAODv12/DYto2Tau-2Jets_MLL-50_2J_TuneCP5_13p6TeV_amcatnloFXFX-pythia8-photos/NANOAODSIM/130X_mcRun3_2022_realistic_postEE_v6-v2/2520000/75aaf464-bf10-47ee-b68f-fa36cd771538.root,1791417247019,redir.t2.ucsd.edu,Production Output,264805364,davs://redir.t2.ucsd.edu:1095/store/mc/Run3Summer22EENanoAODv12/DYto2Tau-2Jets_MLL-50_2J_TuneCP5_13p6Te

In [8]:
output_filename = 'df.csv'
df_cleaned.to_csv(output_filename, index=False, encoding='utf-8-sig')
print(f"FILE_TAG: df.csv")

FILE_TAG: df_cleaned.csv
